# Real-Time Drowsiness Detection System (MediaPipe + Google Colab)

This notebook runs a **browser-based real-time drowsiness detector** powered by **MediaPipe Face Landmarker (WebAssembly / WebGL)** instead of Dlib + Gradio frame streaming.

### Why this architecture fixes the slow/stuck `Monitoring...` behavior
- **Zero frame uploads to Colab:** Webcam frames stay in your browser (`Browser Webcam -> MediaPipe Face Landmarker JS/WASM -> Live Overlay UI`). Colab only serves the static HTML/JS app and provides a public HTTPS tunnel.
- **No Dlib or OpenCV compile needed:** Setup takes seconds and runs smoothly at 15+ FPS using GPU (WebGL) acceleration with automatic CPU fallback.
- **Reliable drowsiness classification:** Uses aspect-ratio-corrected Eye Aspect Ratio (EAR) across **both** eyes, EMA smoothing, hysteresis, head-turn guards, and **time-based thresholds** (ignoring brief natural blinks) to classify **ACTIVE**, **DROWSY**, **SLEEPING**, and **NO FACE** directly over the live video.

> **Disclaimer:** This is an educational drowsiness estimation prototype, not a certified driver-safety or medical monitoring device.

### Quick Start
Run the code cells below from top to bottom:
1. **Environment setup** (installs `cloudflared` tunnel binary if needed)
2. **Write web app & start local server helper**
3. **Launch server & public HTTPS URL** — open the printed `https://...trycloudflare.com` link, allow camera permission, and click **Start Camera**.

In [ ]:
# Step 1: Lightweight environment setup (Cloudflare Tunnel binary only)
# No Dlib, OpenCV, or Gradio installation is required because inference runs in the browser.

import os
import platform
import shutil
import subprocess
import sys
import urllib.request

print(f"Python version: {sys.version.split()[0]}")
print(f"Platform: {platform.system()} ({platform.machine()})")

CLOUDFLARED_PATH = shutil.which("cloudflared") or "/usr/local/bin/cloudflared"

if os.path.exists(CLOUDFLARED_PATH) and os.access(CLOUDFLARED_PATH, os.X_OK):
    print(f"cloudflared already installed at: {CLOUDFLARED_PATH}")
else:
    arch = platform.machine().lower()
    if arch in ("x86_64", "amd64"):
        binary_name = "cloudflared-linux-amd64"
    elif arch in ("aarch64", "arm64"):
        binary_name = "cloudflared-linux-arm64"
    else:
        raise RuntimeError(f"Unsupported CPU architecture for automatic cloudflared download: {arch}")

    download_url = f"https://github.com/cloudflare/cloudflared/releases/latest/download/{binary_name}"
    print(f"Downloading {binary_name} from Cloudflare releases...")
    try:
        urllib.request.urlretrieve(download_url, CLOUDFLARED_PATH)
        os.chmod(CLOUDFLARED_PATH, 0o755)
    except Exception as exc:
        raise RuntimeError(
            f"Failed to download cloudflared from {download_url}. "
            f"Check internet connectivity in the Colab runtime. Error: {exc}"
        ) from exc

version_proc = subprocess.run(
    [CLOUDFLARED_PATH, "--version"],
    capture_output=True,
    text=True,
    check=True,
)
print("cloudflared ready:", version_proc.stdout.strip())

In [ ]:
# Step 2: Create the browser-based MediaPipe web application and HTTP server

import functools
import http.server
import socket
import socketserver
import threading
from pathlib import Path

APP_DIR = Path("/content/drowsiness_app") if Path("/content").exists() else (Path.cwd() / "drowsiness_app")
APP_DIR.mkdir(parents=True, exist_ok=True)
INDEX_FILE = APP_DIR / "index.html"

APP_HTML = r"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>Real-Time Drowsiness Detector (MediaPipe)</title>
  <style>
    :root {
      --bg: #0b1020;
      --panel: #121933;
      --panel-border: #243056;
      --text: #eef2ff;
      --muted: #9aa7c7;
      --active: #16a34a;
      --active-bg: rgba(22, 163, 74, 0.24);
      --drowsy: #ea580c;
      --drowsy-bg: rgba(234, 88, 12, 0.28);
      --sleeping: #dc2626;
      --sleeping-bg: rgba(220, 38, 38, 0.34);
      --noface: #64748b;
      --noface-bg: rgba(100, 116, 139, 0.30);
      --accent: #3b82f6;
    }

    * { box-sizing: border-box; }

    body {
      margin: 0;
      font-family: system-ui, -apple-system, Segoe UI, Roboto, Helvetica, Arial, sans-serif;
      background: radial-gradient(circle at top, #131d3d 0%, var(--bg) 60%);
      color: var(--text);
      min-height: 100vh;
      padding: 16px;
    }

    .container {
      max-width: 1120px;
      margin: 0 auto;
      display: flex;
      flex-direction: column;
      gap: 14px;
    }

    header {
      display: flex;
      flex-wrap: wrap;
      justify-content: space-between;
      align-items: center;
      gap: 10px;
    }

    h1 {
      margin: 0;
      font-size: 1.35rem;
      font-weight: 700;
      letter-spacing: -0.01em;
    }

    .subtitle {
      margin: 2px 0 0;
      font-size: 0.85rem;
      color: var(--muted);
    }

    .badge-row {
      display: flex;
      flex-wrap: wrap;
      gap: 8px;
      align-items: center;
    }

    .pill {
      padding: 5px 10px;
      border-radius: 999px;
      font-size: 0.78rem;
      font-weight: 600;
      background: var(--panel);
      border: 1px solid var(--panel-border);
      color: var(--muted);
    }

    .pill strong { color: var(--text); }

    .error-banner {
      display: none;
      padding: 12px 14px;
      border-radius: 10px;
      background: rgba(220, 38, 38, 0.16);
      border: 1px solid rgba(248, 113, 113, 0.5);
      color: #fecaca;
      font-size: 0.9rem;
      line-height: 1.4;
    }
    .error-banner.visible { display: block; }

    .main-grid {
      display: grid;
      grid-template-columns: minmax(0, 1.7fr) minmax(290px, 1fr);
      gap: 14px;
      align-items: start;
    }

    @media (max-width: 900px) {
      .main-grid { grid-template-columns: 1fr; }
    }

    .video-card {
      background: var(--panel);
      border: 2px solid var(--panel-border);
      border-radius: 14px;
      overflow: hidden;
      transition: border-color 0.2s ease, box-shadow 0.2s ease;
    }

    .video-card.state-ACTIVE {
      border-color: var(--active);
      box-shadow: 0 0 24px rgba(22, 163, 74, 0.22);
    }
    .video-card.state-DROWSY {
      border-color: var(--drowsy);
      box-shadow: 0 0 28px rgba(234, 88, 12, 0.30);
    }
    .video-card.state-SLEEPING {
      border-color: var(--sleeping);
      box-shadow: 0 0 34px rgba(220, 38, 38, 0.45);
      animation: pulseBorder 0.9s infinite alternate;
    }
    .video-card.state-NO_FACE {
      border-color: var(--noface);
    }

    @keyframes pulseBorder {
      from { box-shadow: 0 0 18px rgba(220, 38, 38, 0.35); }
      to { box-shadow: 0 0 38px rgba(220, 38, 38, 0.72); }
    }

    .video-stage {
      position: relative;
      width: 100%;
      aspect-ratio: 16 / 9;
      background: #050812;
      overflow: hidden;
    }

    #webcam, #overlayCanvas {
      position: absolute;
      inset: 0;
      width: 100%;
      height: 100%;
      object-fit: cover;
    }

    .mirrored {
      transform: scaleX(-1);
    }

    #overlayCanvas {
      pointer-events: none;
    }

    /* Non-mirrored live HUD rendered directly over the webcam video */
    .hud-layer {
      position: absolute;
      inset: 0;
      pointer-events: none;
      display: flex;
      flex-direction: column;
      justify-content: space-between;
      padding: 12px;
    }

    .hud-top {
      display: flex;
      justify-content: space-between;
      align-items: flex-start;
      gap: 10px;
      flex-wrap: wrap;
    }

    .status-banner {
      display: inline-flex;
      align-items: center;
      gap: 10px;
      padding: 10px 18px;
      border-radius: 12px;
      font-size: clamp(1.15rem, 2.4vw, 1.65rem);
      font-weight: 800;
      letter-spacing: 0.04em;
      color: #ffffff;
      background: rgba(15, 23, 42, 0.80);
      border: 2px solid rgba(255, 255, 255, 0.25);
      backdrop-filter: blur(8px);
      text-shadow: 0 1px 3px rgba(0, 0, 0, 0.65);
    }

    .status-dot {
      width: 14px;
      height: 14px;
      border-radius: 50%;
      background: #94a3b8;
      box-shadow: 0 0 10px currentColor;
      flex-shrink: 0;
    }

    .status-banner.ACTIVE {
      background: var(--active-bg);
      border-color: #4ade80;
    }
    .status-banner.ACTIVE .status-dot { background: #4ade80; }

    .status-banner.DROWSY {
      background: var(--drowsy-bg);
      border-color: #fb923c;
    }
    .status-banner.DROWSY .status-dot { background: #fb923c; }

    .status-banner.SLEEPING {
      background: var(--sleeping-bg);
      border-color: #f87171;
    }
    .status-banner.SLEEPING .status-dot { background: #f87171; }

    .status-banner.NO_FACE {
      background: var(--noface-bg);
      border-color: #94a3b8;
    }
    .status-banner.NO_FACE .status-dot { background: #94a3b8; }

    .hud-subnote {
      padding: 6px 11px;
      border-radius: 8px;
      font-size: 0.8rem;
      font-weight: 600;
      background: rgba(15, 23, 42, 0.78);
      border: 1px solid rgba(255, 255, 255, 0.16);
      color: #fde68a;
      backdrop-filter: blur(6px);
      display: none;
    }
    .hud-subnote.visible { display: inline-block; }

    .hud-bottom {
      display: grid;
      grid-template-columns: repeat(4, minmax(0, 1fr));
      gap: 8px;
    }

    @media (max-width: 640px) {
      .hud-bottom { grid-template-columns: repeat(2, minmax(0, 1fr)); }
    }

    .hud-metric {
      background: rgba(9, 14, 29, 0.76);
      border: 1px solid rgba(255, 255, 255, 0.14);
      border-radius: 10px;
      padding: 7px 10px;
      backdrop-filter: blur(6px);
    }

    .hud-metric-label {
      font-size: 0.7rem;
      text-transform: uppercase;
      letter-spacing: 0.05em;
      color: #cbd5e1;
    }

    .hud-metric-value {
      font-size: 1.05rem;
      font-weight: 700;
      font-variant-numeric: tabular-nums;
      margin-top: 2px;
      color: #ffffff;
    }

    .placeholder-overlay {
      position: absolute;
      inset: 0;
      display: flex;
      flex-direction: column;
      align-items: center;
      justify-content: center;
      text-align: center;
      padding: 20px;
      background: rgba(6, 10, 22, 0.75);
      color: var(--muted);
      gap: 8px;
    }
    .placeholder-overlay.hidden { display: none; }

    .controls-bar {
      display: flex;
      flex-wrap: wrap;
      gap: 10px;
      padding: 12px;
      background: rgba(9, 13, 28, 0.7);
      border-top: 1px solid var(--panel-border);
      align-items: center;
    }

    button {
      appearance: none;
      border: none;
      border-radius: 9px;
      padding: 10px 16px;
      font-size: 0.9rem;
      font-weight: 600;
      cursor: pointer;
      transition: transform 0.08s ease, opacity 0.15s ease, background 0.15s ease;
    }
    button:active:not(:disabled) { transform: scale(0.98); }
    button:disabled { opacity: 0.45; cursor: not-allowed; }

    .btn-primary { background: #2563eb; color: #fff; }
    .btn-primary:hover:not(:disabled) { background: #1d4ed8; }

    .btn-danger { background: #dc2626; color: #fff; }
    .btn-danger:hover:not(:disabled) { background: #b91c1c; }

    .btn-secondary {
      background: #1e294b;
      color: var(--text);
      border: 1px solid var(--panel-border);
    }
    .btn-secondary:hover:not(:disabled) { background: #293763; }

    .btn-muted {
      background: #7c2d12;
      color: #fed7aa;
      border: 1px solid #ea580c;
    }

    .side-panel {
      background: var(--panel);
      border: 1px solid var(--panel-border);
      border-radius: 14px;
      padding: 14px;
      display: flex;
      flex-direction: column;
      gap: 12px;
    }

    .side-panel h2 {
      margin: 0;
      font-size: 1rem;
      font-weight: 700;
    }

    .setting-row {
      display: flex;
      flex-direction: column;
      gap: 4px;
    }

    .setting-header {
      display: flex;
      justify-content: space-between;
      font-size: 0.82rem;
    }

    .setting-header label { color: var(--text); font-weight: 600; }
    .setting-header span { color: #93c5fd; font-variant-numeric: tabular-nums; font-weight: 600; }

    .setting-hint {
      font-size: 0.74rem;
      color: var(--muted);
      line-height: 1.3;
    }

    input[type="range"] {
      width: 100%;
      accent-color: var(--accent);
      cursor: pointer;
    }

    .checkbox-row {
      display: flex;
      align-items: center;
      justify-content: space-between;
      gap: 8px;
      font-size: 0.85rem;
      padding: 4px 0;
    }

    .legend {
      border-top: 1px solid var(--panel-border);
      padding-top: 10px;
      font-size: 0.78rem;
      color: var(--muted);
      line-height: 1.45;
    }
    .legend strong { color: var(--text); }

    .disclaimer {
      font-size: 0.78rem;
      color: var(--muted);
      text-align: center;
      padding: 4px 0 8px;
    }
  </style>
</head>
<body>
  <div class="container">
    <header>
      <div>
        <h1>Real-Time Drowsiness Detection</h1>
        <p class="subtitle">Local browser inference via MediaPipe Face Landmarker (no video sent to Colab)</p>
      </div>
      <div class="badge-row">
        <span class="pill">Model: <strong id="modelStateBadge">Not loaded</strong></span>
        <span class="pill">Delegate: <strong id="delegateBadge">—</strong></span>
        <span class="pill">Alarm: <strong id="alarmStateBadge">Ready</strong></span>
      </div>
    </header>

    <div id="errorBanner" class="error-banner" role="alert"></div>

    <div class="main-grid">
      <section id="videoCard" class="video-card">
        <div class="video-stage">
          <video id="webcam" class="mirrored" autoplay playsinline muted></video>
          <canvas id="overlayCanvas" class="mirrored"></canvas>

          <div class="hud-layer">
            <div class="hud-top">
              <div id="statusBanner" class="status-banner NO_FACE">
                <span class="status-dot"></span>
                <span id="statusText">IDLE</span>
              </div>
              <div id="hudSubnote" class="hud-subnote"></div>
            </div>

            <div class="hud-bottom">
              <div class="hud-metric">
                <div class="hud-metric-label">Left Eye EAR</div>
                <div id="leftEarValue" class="hud-metric-value">—</div>
              </div>
              <div class="hud-metric">
                <div class="hud-metric-label">Right Eye EAR</div>
                <div id="rightEarValue" class="hud-metric-value">—</div>
              </div>
              <div class="hud-metric">
                <div class="hud-metric-label">Closure Duration</div>
                <div id="closureDurationValue" class="hud-metric-value">0.00 s</div>
              </div>
              <div class="hud-metric">
                <div class="hud-metric-label">Processing FPS</div>
                <div id="fpsValue" class="hud-metric-value">0.0 FPS</div>
              </div>
            </div>
          </div>

          <div id="placeholderOverlay" class="placeholder-overlay">
            <strong style="color:#eef2ff;font-size:1.05rem;">Camera is stopped</strong>
            <span>Click <b>Start Camera</b> below and allow webcam access to start real-time detection.</span>
          </div>
        </div>

        <div class="controls-bar">
          <button id="startBtn" class="btn-primary">Start Camera</button>
          <button id="stopBtn" class="btn-danger" disabled>Stop Camera</button>
          <button id="muteBtn" class="btn-secondary">Mute Alarm</button>
          <button id="testAlarmBtn" class="btn-secondary">Test Sound</button>
          <button id="calibrateBtn" class="btn-secondary" disabled>Calibrate Open Eyes</button>
        </div>
      </section>

      <aside class="side-panel">
        <h2>Detection Calibration</h2>

        <div class="setting-row">
          <div class="setting-header">
            <label for="earClosedSlider">Closed EAR Threshold</label>
            <span id="earClosedVal">0.19</span>
          </div>
          <input id="earClosedSlider" type="range" min="0.12" max="0.26" step="0.01" value="0.19" />
          <div class="setting-hint">Both eyes below this value count as closed (triggers SLEEPING timer).</div>
        </div>

        <div class="setting-row">
          <div class="setting-header">
            <label for="earDrowsySlider">Drowsy EAR Threshold</label>
            <span id="earDrowsyVal">0.24</span>
          </div>
          <input id="earDrowsySlider" type="range" min="0.18" max="0.32" step="0.01" value="0.24" />
          <div class="setting-hint">Both eyes below this value count as partially closed (triggers DROWSY timer).</div>
        </div>

        <div class="setting-row">
          <div class="setting-header">
            <label for="drowsyDurSlider">Drowsy Duration</label>
            <span id="drowsyDurVal">1.2 s</span>
          </div>
          <input id="drowsyDurSlider" type="range" min="0.5" max="4.0" step="0.1" value="1.2" />
          <div class="setting-hint">Exceeds normal blinks (~0.15–0.40 s) before flagging DROWSY.</div>
        </div>

        <div class="setting-row">
          <div class="setting-header">
            <label for="sleepDurSlider">Sleeping Duration</label>
            <span id="sleepDurVal">2.5 s</span>
          </div>
          <input id="sleepDurSlider" type="range" min="1.0" max="8.0" step="0.1" value="2.5" />
          <div class="setting-hint">Sustained full closure of both eyes required to trigger SLEEPING & alarm.</div>
        </div>

        <div class="setting-row">
          <div class="setting-header">
            <label for="targetFpsSlider">Inference Rate Cap</label>
            <span id="targetFpsVal">15 FPS</span>
          </div>
          <input id="targetFpsSlider" type="range" min="5" max="30" step="1" value="15" />
          <div class="setting-hint">Limits MediaPipe inference frequency to keep UI smooth and battery friendly.</div>
        </div>

        <div class="setting-row">
          <div class="setting-header">
            <label for="cooldownSlider">Alarm Cooldown</label>
            <span id="cooldownVal">5.0 s</span>
          </div>
          <input id="cooldownSlider" type="range" min="2.0" max="20.0" step="0.5" value="5.0" />
          <div class="setting-hint">Minimum wait between repeated alarm alerts while remaining SLEEPING.</div>
        </div>

        <div class="checkbox-row">
          <label for="showOverlayCheck">Show face box & eye landmarks</label>
          <input id="showOverlayCheck" type="checkbox" checked />
        </div>

        <div class="checkbox-row">
          <label for="mirrorVideoCheck">Mirror camera preview</label>
          <input id="mirrorVideoCheck" type="checkbox" checked />
        </div>

        <button id="resetDefaultsBtn" class="btn-secondary" style="width:100%;">Reset Default Thresholds</button>

        <div class="legend">
          <strong>Status Guide:</strong><br />
          <span style="color:#4ade80;">● ACTIVE</span>: Face tracked, eyes open.<br />
          <span style="color:#fb923c;">● DROWSY</span>: Sustained partial or slow eye closure.<br />
          <span style="color:#f87171;">● SLEEPING</span>: Both eyes closed past sleep threshold.<br />
          <span style="color:#94a3b8;">● NO FACE</span>: Face absent past 0.5 s grace window.
        </div>
      </aside>
    </div>

    <div class="disclaimer">
      Educational drowsiness estimation prototype using MediaPipe Face Landmarker. Not a certified driver-safety or medical monitoring system.
    </div>
  </div>

  <script type="module">
    // -------------------------------------------------------------------------
    // MediaPipe CDN & Model Configuration (pinned to verified @1.2.3 release)
    // -------------------------------------------------------------------------
    const MEDIAPIPE_BUNDLE_URL =
      "https://cdn.jsdelivr.net/npm/@mediapipe/tasks-vision@1.2.3/vision_bundle.mjs";
    const MEDIAPIPE_WASM_URL =
      "https://cdn.jsdelivr.net/npm/@mediapipe/tasks-vision@1.2.3/wasm";
    const FACE_LANDMARKER_MODEL_URL =
      "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task";

    // Verified against @mediapipe/tasks-vision@1.2.3 FaceLandmarker 478-point mesh:
    // Left Eye = subject's anatomical left eye; Right Eye = subject's anatomical right eye.
    const LEFT_EYE = {
      outer: 263,
      inner: 362,
      upper: [387, 386, 385],
      lower: [373, 374, 380],
      contour: [263, 466, 388, 387, 386, 385, 384, 398, 362, 382, 381, 380, 374, 373, 390, 249],
      irisCenter: 473
    };

    const RIGHT_EYE = {
      outer: 33,
      inner: 133,
      upper: [160, 159, 158],
      lower: [144, 145, 153],
      contour: [33, 246, 161, 160, 159, 158, 157, 173, 133, 155, 154, 153, 145, 144, 163, 7],
      irisCenter: 468
    };

    const NOSE_TIP_INDEX = 1;

    // -------------------------------------------------------------------------
    // Default Tunable Parameters
    // -------------------------------------------------------------------------
    const DEFAULTS = {
      earClosedThreshold: 0.19,
      earDrowsyThreshold: 0.24,
      earHysteresis: 0.02,
      drowsyDurationSec: 1.2,
      sleepDurationSec: 2.5,
      targetFps: 15,
      alarmCooldownSec: 5.0,
      maxYawAsymmetry: 0.48,
      emaAlpha: 0.45,
      faceLostGraceMs: 500,
      recoveryGraceMs: 350
    };

    const config = { ...DEFAULTS };

    // -------------------------------------------------------------------------
    // DOM Elements
    // -------------------------------------------------------------------------
    const videoEl = document.getElementById("webcam");
    const canvasEl = document.getElementById("overlayCanvas");
    const ctx = canvasEl.getContext("2d");
    const videoCardEl = document.getElementById("videoCard");
    const placeholderEl = document.getElementById("placeholderOverlay");
    const errorBannerEl = document.getElementById("errorBanner");

    const statusBannerEl = document.getElementById("statusBanner");
    const statusTextEl = document.getElementById("statusText");
    const hudSubnoteEl = document.getElementById("hudSubnote");
    const leftEarEl = document.getElementById("leftEarValue");
    const rightEarEl = document.getElementById("rightEarValue");
    const closureDurationEl = document.getElementById("closureDurationValue");
    const fpsEl = document.getElementById("fpsValue");

    const modelStateBadgeEl = document.getElementById("modelStateBadge");
    const delegateBadgeEl = document.getElementById("delegateBadge");
    const alarmStateBadgeEl = document.getElementById("alarmStateBadge");

    const startBtn = document.getElementById("startBtn");
    const stopBtn = document.getElementById("stopBtn");
    const muteBtn = document.getElementById("muteBtn");
    const testAlarmBtn = document.getElementById("testAlarmBtn");
    const calibrateBtn = document.getElementById("calibrateBtn");
    const resetDefaultsBtn = document.getElementById("resetDefaultsBtn");

    const earClosedSlider = document.getElementById("earClosedSlider");
    const earDrowsySlider = document.getElementById("earDrowsySlider");
    const drowsyDurSlider = document.getElementById("drowsyDurSlider");
    const sleepDurSlider = document.getElementById("sleepDurSlider");
    const targetFpsSlider = document.getElementById("targetFpsSlider");
    const cooldownSlider = document.getElementById("cooldownSlider");

    const earClosedValEl = document.getElementById("earClosedVal");
    const earDrowsyValEl = document.getElementById("earDrowsyVal");
    const drowsyDurValEl = document.getElementById("drowsyDurVal");
    const sleepDurValEl = document.getElementById("sleepDurVal");
    const targetFpsValEl = document.getElementById("targetFpsVal");
    const cooldownValEl = document.getElementById("cooldownVal");

    const showOverlayCheck = document.getElementById("showOverlayCheck");
    const mirrorVideoCheck = document.getElementById("mirrorVideoCheck");

    // -------------------------------------------------------------------------
    // Utility Helpers
    // -------------------------------------------------------------------------
    function clamp(val, min, max) {
      return Math.min(max, Math.max(min, val));
    }

    function showError(message) {
      errorBannerEl.textContent = message;
      errorBannerEl.classList.add("visible");
    }

    function clearError() {
      errorBannerEl.textContent = "";
      errorBannerEl.classList.remove("visible");
    }

    function dist2D(a, b) {
      const dx = a.x - b.x;
      const dy = a.y - b.y;
      return Math.hypot(dx, dy);
    }

    /**
     * Compute aspect-ratio-corrected Eye Aspect Ratio (EAR) for one eye.
     * Multiplying normalized coordinates by frame pixel dimensions ensures
     * vertical and horizontal distances are measured on the same scale.
     */
    function computeEyeMetrics(landmarks, eyeDef, frameWidth, frameHeight) {
      const toPx = (idx) => {
        const pt = landmarks[idx];
        if (!pt || !Number.isFinite(pt.x) || !Number.isFinite(pt.y)) return null;
        return { x: pt.x * frameWidth, y: pt.y * frameHeight };
      };

      const outer = toPx(eyeDef.outer);
      const inner = toPx(eyeDef.inner);
      if (!outer || !inner) {
        return { valid: false, ear: null, widthPx: 0 };
      }

      const horiz = dist2D(outer, inner);
      const minEyeWidthPx = Math.max(10, frameWidth * 0.012);
      if (!Number.isFinite(horiz) || horiz < minEyeWidthPx) {
        return { valid: false, ear: null, widthPx: horiz || 0 };
      }

      let vertSum = 0;
      for (let i = 0; i < eyeDef.upper.length; i++) {
        const up = toPx(eyeDef.upper[i]);
        const lo = toPx(eyeDef.lower[i]);
        if (!up || !lo) {
          return { valid: false, ear: null, widthPx: horiz };
        }
        vertSum += dist2D(up, lo);
      }

      const ear = (vertSum / eyeDef.upper.length) / horiz;
      if (!Number.isFinite(ear) || ear < 0 || ear > 0.75) {
        return { valid: false, ear: null, widthPx: horiz };
      }

      return { valid: true, ear, widthPx: horiz };
    }

    /**
     * Estimate horizontal head turn (yaw asymmetry) from nose tip and outer eye corners.
     * Returns value in [0, 1]; near 0 when facing camera, > 0.48 when turned sharply sideways.
     */
    function computeYawAsymmetry(landmarks, frameWidth, frameHeight) {
      const nose = landmarks[NOSE_TIP_INDEX];
      const leftOuter = landmarks[LEFT_EYE.outer];
      const rightOuter = landmarks[RIGHT_EYE.outer];
      if (!nose || !leftOuter || !rightOuter) return 1.0;

      const nPx = { x: nose.x * frameWidth, y: nose.y * frameHeight };
      const lPx = { x: leftOuter.x * frameWidth, y: leftOuter.y * frameHeight };
      const rPx = { x: rightOuter.x * frameWidth, y: rightOuter.y * frameHeight };

      const dLeft = dist2D(nPx, lPx);
      const dRight = dist2D(nPx, rPx);
      const total = dLeft + dRight;
      if (!Number.isFinite(total) || total < 20) return 1.0;

      return Math.abs(dLeft - dRight) / total;
    }

    // -------------------------------------------------------------------------
    // Web Audio Alarm Controller (Browser-Safe Autoplay + Cooldown)
    // -------------------------------------------------------------------------
    class AlarmController {
      constructor() {
        this.ctx = null;
        this.muted = false;
        this.lastTriggerMs = 0;
        this.activeNodes = [];
        this.activeTimer = null;
      }

      /** Must be called synchronously inside a user click gesture to satisfy browser autoplay rules. */
      unlock() {
        try {
          if (!this.ctx) {
            const AudioCtx = window.AudioContext || window.webkitAudioContext;
            if (!AudioCtx) return false;
            this.ctx = new AudioCtx();
          }
          if (this.ctx.state === "suspended") {
            this.ctx.resume().catch(() => {});
          }
          return true;
        } catch {
          return false;
        }
      }

      setMuted(muted) {
        this.muted = muted;
        if (muted) {
          this.stopImmediate();
        }
        this.updateBadge();
      }

      updateBadge() {
        if (this.muted) {
          alarmStateBadgeEl.textContent = "Muted";
          muteBtn.textContent = "Unmute Alarm";
          muteBtn.className = "btn-muted";
        } else {
          alarmStateBadgeEl.textContent = "Armed";
          muteBtn.textContent = "Mute Alarm";
          muteBtn.className = "btn-secondary";
        }
      }

      resetCooldown() {
        this.lastTriggerMs = 0;
      }

      stopImmediate() {
        if (this.activeTimer) {
          clearTimeout(this.activeTimer);
          this.activeTimer = null;
        }
        for (const node of this.activeNodes) {
          try { node.stop(); } catch {}
          try { node.disconnect(); } catch {}
        }
        this.activeNodes = [];
      }

      /** Play alarm only if unmuted and cooldown has elapsed. */
      triggerIfDue(nowMs, force = false) {
        if (this.muted && !force) return false;
        const cooldownMs = config.alarmCooldownSec * 1000;
        if (!force && this.lastTriggerMs > 0 && (nowMs - this.lastTriggerMs) < cooldownMs) {
          return false;
        }
        if (!this.unlock() || !this.ctx) return false;

        this.lastTriggerMs = nowMs;
        this._playDualToneBurst();
        return true;
      }

      _playDualToneBurst() {
        this.stopImmediate();
        const now = this.ctx.currentTime;
        const pulses = [
          { offset: 0.0, dur: 0.22, freq: 880 },
          { offset: 0.28, dur: 0.22, freq: 1046.5 },
          { offset: 0.56, dur: 0.26, freq: 880 }
        ];

        for (const p of pulses) {
          const osc = this.ctx.createOscillator();
          const gain = this.ctx.createGain();
          osc.type = "triangle";
          osc.frequency.setValueAtTime(p.freq, now + p.offset);

          gain.gain.setValueAtTime(0.0001, now + p.offset);
          gain.gain.exponentialRampToValueAtTime(0.24, now + p.offset + 0.02);
          gain.gain.exponentialRampToValueAtTime(0.0001, now + p.offset + p.dur);

          osc.connect(gain);
          gain.connect(this.ctx.destination);
          osc.start(now + p.offset);
          osc.stop(now + p.offset + p.dur + 0.02);
          this.activeNodes.push(osc);
        }

        this.activeTimer = setTimeout(() => {
          this.activeNodes = [];
          this.activeTimer = null;
        }, 950);
      }
    }

    // -------------------------------------------------------------------------
    // Time-Based Drowsiness Classification State Machine
    // -------------------------------------------------------------------------
    class DrowsinessClassifier {
      constructor() {
        this.reset();
      }

      reset() {
        this.state = "IDLE";
        this.smoothLeftEar = null;
        this.smoothRightEar = null;
        this.closedDurationMs = 0;
        this.drowsyDurationMs = 0;
        this.recoveryDurationMs = 0;
        this.lastFaceSeenMs = 0;
        this.lastTickMs = 0;
        this.wasClosed = false;
        this.wasDrowsy = false;
        this.subnote = "";
      }

      /**
       * Advance state using real elapsed wall-clock time (dt) rather than frame counts.
       * Requires BOTH eyes to confirm closure/drowsiness so a single-eye landmark glitch
       * or wink cannot trigger SLEEPING.
       */
      update({ nowMs, faceFound, leftMetrics, rightMetrics, yawAsymmetry }) {
        if (this.lastTickMs === 0) {
          this.lastTickMs = nowMs;
        }
        // Cap dt so browser tab throttling doesn't create a sudden false jump.
        const dt = clamp(nowMs - this.lastTickMs, 0, 250);
        this.lastTickMs = nowMs;

        // 1. Handle missing face with short grace period to avoid 1-frame flicker.
        if (!faceFound) {
          this.subnote = "";
          if (this.state === "IDLE") {
            this.state = "NO_FACE";
          } else if (this.lastFaceSeenMs === 0 || (nowMs - this.lastFaceSeenMs) >= config.faceLostGraceMs) {
            this.state = "NO_FACE";
            this.closedDurationMs = 0;
            this.drowsyDurationMs = 0;
            this.recoveryDurationMs = 0;
            this.smoothLeftEar = null;
            this.smoothRightEar = null;
            this.wasClosed = false;
            this.wasDrowsy = false;
          }
          return this.snapshot();
        }

        this.lastFaceSeenMs = nowMs;

        // 2. Guard against sharp head turns ("looking away") where far eye foreshortens.
        if (yawAsymmetry > config.maxYawAsymmetry) {
          this.subnote = "Head turned — look toward camera";
          this.closedDurationMs = Math.max(0, this.closedDurationMs - dt * 0.5);
          this.drowsyDurationMs = Math.max(0, this.drowsyDurationMs - dt * 0.5);
          if (this.state === "IDLE" || this.state === "NO_FACE") {
            this.state = "ACTIVE";
          }
          return this.snapshot();
        }

        // 3. Require both eyes to have geometrically valid landmarks.
        if (!leftMetrics.valid || !rightMetrics.valid) {
          this.subnote = "Low eye landmark confidence";
          if (this.state === "IDLE" || this.state === "NO_FACE") {
            this.state = "ACTIVE";
          }
          return this.snapshot();
        }

        this.subnote = "";

        // 4. Exponential moving average (EMA) smoothing per eye to reduce landmark jitter.
        const a = config.emaAlpha;
        this.smoothLeftEar = this.smoothLeftEar === null
          ? leftMetrics.ear
          : a * leftMetrics.ear + (1 - a) * this.smoothLeftEar;
        this.smoothRightEar = this.smoothRightEar === null
          ? rightMetrics.ear
          : a * rightMetrics.ear + (1 - a) * this.smoothRightEar;

        const lEar = this.smoothLeftEar;
        const rEar = this.smoothRightEar;

        // 5. Hysteresis thresholds prevent flickering around boundary values.
        const closedEnter = config.earClosedThreshold;
        const closedExit = closedEnter + config.earHysteresis;
        const drowsyEnter = Math.max(config.earDrowsyThreshold, closedEnter + 0.02);
        const drowsyExit = drowsyEnter + config.earHysteresis;

        const leftClosed = lEar < (this.wasClosed ? closedExit : closedEnter);
        const rightClosed = rEar < (this.wasClosed ? closedExit : closedEnter);
        const bothClosed = leftClosed && rightClosed;

        const leftDrowsy = lEar < (this.wasDrowsy ? drowsyExit : drowsyEnter);
        const rightDrowsy = rEar < (this.wasDrowsy ? drowsyExit : drowsyEnter);
        const bothDrowsyOrClosed = leftDrowsy && rightDrowsy;

        this.wasClosed = bothClosed;
        this.wasDrowsy = bothDrowsyOrClosed;

        // 6. Accumulate time-based durations.
        if (bothClosed) {
          this.closedDurationMs += dt;
          this.drowsyDurationMs += dt;
          this.recoveryDurationMs = 0;
        } else if (bothDrowsyOrClosed) {
          // Partially closed: advance drowsy timer, decay full-closed timer cleanly.
          this.drowsyDurationMs += dt;
          this.closedDurationMs = Math.max(0, this.closedDurationMs - dt * 2.0);
          this.recoveryDurationMs = 0;
        } else {
          // Eyes open normally.
          this.recoveryDurationMs += dt;
          if (this.recoveryDurationMs >= config.recoveryGraceMs || this.state === "ACTIVE" || this.state === "NO_FACE" || this.state === "IDLE") {
            this.closedDurationMs = 0;
            this.drowsyDurationMs = 0;
          }
        }

        // 7. Classify state from elapsed durations.
        const drowsyMs = config.drowsyDurationSec * 1000;
        const sleepMs = Math.max(config.sleepDurationSec * 1000, drowsyMs + 400);

        if (this.closedDurationMs >= sleepMs) {
          this.state = "SLEEPING";
        } else if (this.drowsyDurationMs >= drowsyMs) {
          // Stay SLEEPING until recoveryGraceMs is met so 1-frame twitch doesn't bounce.
          if (this.state === "SLEEPING" && this.recoveryDurationMs < config.recoveryGraceMs) {
            this.state = "SLEEPING";
          } else {
            this.state = "DROWSY";
          }
        } else if (
          (this.state === "SLEEPING" || this.state === "DROWSY") &&
          this.recoveryDurationMs < config.recoveryGraceMs
        ) {
          // Hold DROWSY/SLEEPING briefly until open eyes are sustained for recoveryGraceMs.
        } else {
          this.state = "ACTIVE";
        }

        return this.snapshot();
      }

      snapshot() {
        return {
          state: this.state,
          leftEar: this.smoothLeftEar,
          rightEar: this.smoothRightEar,
          closureSec: Math.max(this.closedDurationMs, this.drowsyDurationMs) / 1000,
          subnote: this.subnote
        };
      }
    }

    // -------------------------------------------------------------------------
    // MediaPipe Loader & Runtime State
    // -------------------------------------------------------------------------
    let faceLandmarker = null;
    let activeDelegate = "—";
    let mediaStream = null;
    let rafId = null;
    let isRunning = false;
    let isStarting = false;
    let sessionToken = 0;

    let lastVideoTime = -1;
    let lastInferMs = 0;
    let fpsCounter = 0;
    let fpsWindowStartMs = 0;
    let currentFps = 0;

    let calibrationActive = false;
    let calibrationSamples = [];
    let calibrationEndMs = 0;

    const alarm = new AlarmController();
    const classifier = new DrowsinessClassifier();

    async function ensureFaceLandmarker() {
      if (faceLandmarker) return faceLandmarker;

      modelStateBadgeEl.textContent = "Loading WASM & model...";
      let visionModule;
      try {
        visionModule = await import(MEDIAPIPE_BUNDLE_URL);
      } catch (err) {
        modelStateBadgeEl.textContent = "CDN load failed";
        throw new Error(
          "Could not load MediaPipe tasks-vision from jsDelivr CDN. Check internet access or firewall/ad-blocker settings. (" +
          (err?.message || err) + ")"
        );
      }

      const { FilesetResolver, FaceLandmarker } = visionModule;
      let fileset;
      try {
        fileset = await FilesetResolver.forVisionTasks(MEDIAPIPE_WASM_URL);
      } catch (err) {
        modelStateBadgeEl.textContent = "WASM load failed";
        throw new Error(
          "Failed to initialize MediaPipe WASM binaries. (" + (err?.message || err) + ")"
        );
      }

      const baseModelConfig = {
        runningMode: "VIDEO",
        numFaces: 1,
        minFaceDetectionConfidence: 0.5,
        minFacePresenceConfidence: 0.5,
        minTrackingConfidence: 0.5,
        outputFaceBlendshapes: false,
        outputFacialTransformationMatrixes: false
      };

      // Prefer GPU delegate, automatically falling back to CPU if WebGL/GPU is unavailable.
      for (const delegate of ["GPU", "CPU"]) {
        try {
          faceLandmarker = await FaceLandmarker.createFromOptions(fileset, {
            baseOptions: {
              modelAssetPath: FACE_LANDMARKER_MODEL_URL,
              delegate
            },
            ...baseModelConfig
          });
          activeDelegate = delegate;
          delegateBadgeEl.textContent = delegate === "GPU" ? "GPU (WebGL)" : "CPU (fallback)";
          modelStateBadgeEl.textContent = "Ready";
          return faceLandmarker;
        } catch (err) {
          console.warn(`FaceLandmarker init failed on ${delegate}:`, err);
          if (delegate === "CPU") {
            modelStateBadgeEl.textContent = "Model load failed";
            throw new Error(
              "Failed to load MediaPipe FaceLandmarker model on both GPU and CPU. (" +
              (err?.message || err) + ")"
            );
          }
        }
      }
    }

    async function fallbackToCpuDelegate() {
      if (!faceLandmarker || activeDelegate === "CPU") return false;
      try {
        faceLandmarker.close();
      } catch {}
      faceLandmarker = null;
      const visionModule = await import(MEDIAPIPE_BUNDLE_URL);
      const fileset = await visionModule.FilesetResolver.forVisionTasks(MEDIAPIPE_WASM_URL);
      faceLandmarker = await visionModule.FaceLandmarker.createFromOptions(fileset, {
        baseOptions: {
          modelAssetPath: FACE_LANDMARKER_MODEL_URL,
          delegate: "CPU"
        },
        runningMode: "VIDEO",
        numFaces: 1,
        minFaceDetectionConfidence: 0.5,
        minFacePresenceConfidence: 0.5,
        minTrackingConfidence: 0.5
      });
      activeDelegate = "CPU";
      delegateBadgeEl.textContent = "CPU (fallback)";
      return true;
    }

    // -------------------------------------------------------------------------
    // Canvas Overlay Drawing (Face Box + Eye Landmarks)
    // -------------------------------------------------------------------------
    function drawOverlay(landmarks, state) {
      const w = canvasEl.width;
      const h = canvasEl.height;
      ctx.clearRect(0, 0, w, h);

      if (!landmarks || !showOverlayCheck.checked) return;

      const colorMap = {
        ACTIVE: "#4ade80",
        DROWSY: "#fb923c",
        SLEEPING: "#f87171",
        NO_FACE: "#94a3b8"
      };
      const strokeColor = colorMap[state] || "#4ade80";

      // Compute face bounding box from all landmarks.
      let minX = 1, minY = 1, maxX = 0, maxY = 0;
      for (const pt of landmarks) {
        if (pt.x < minX) minX = pt.x;
        if (pt.y < minY) minY = pt.y;
        if (pt.x > maxX) maxX = pt.x;
        if (pt.y > maxY) maxY = pt.y;
      }

      const padX = (maxX - minX) * 0.06 * w;
      const padY = (maxY - minY) * 0.08 * h;
      const bx = clamp(minX * w - padX, 0, w);
      const by = clamp(minY * h - padY, 0, h);
      const bw = clamp((maxX - minX) * w + 2 * padX, 0, w - bx);
      const bh = clamp((maxY - minY) * h + 2 * padY, 0, h - by);

      ctx.strokeStyle = strokeColor;
      ctx.lineWidth = Math.max(2, w / 320);
      ctx.strokeRect(bx, by, bw, bh);

      // Draw left and right eye contours.
      const drawEyeContour = (eyeDef) => {
        ctx.beginPath();
        eyeDef.contour.forEach((idx, i) => {
          const p = landmarks[idx];
          if (!p) return;
          const x = p.x * w;
          const y = p.y * h;
          if (i === 0) ctx.moveTo(x, y);
          else ctx.lineTo(x, y);
        });
        ctx.closePath();
        ctx.strokeStyle = strokeColor;
        ctx.lineWidth = Math.max(1.5, w / 420);
        ctx.stroke();

        // Draw vertical EAR measurement points.
        ctx.fillStyle = "#ffffff";
        for (const idx of [...eyeDef.upper, ...eyeDef.lower, eyeDef.outer, eyeDef.inner]) {
          const p = landmarks[idx];
          if (!p) continue;
          ctx.beginPath();
          ctx.arc(p.x * w, p.y * h, Math.max(2, w / 300), 0, Math.PI * 2);
          ctx.fill();
        }
      };

      drawEyeContour(LEFT_EYE);
      drawEyeContour(RIGHT_EYE);
    }

    function updateUiFromSnapshot(snap, wasActiveReset) {
      const displayLabel = {
        IDLE: "IDLE",
        ACTIVE: "ACTIVE",
        DROWSY: "DROWSY",
        SLEEPING: "SLEEPING",
        NO_FACE: "NO FACE"
      }[snap.state] || snap.state;

      statusTextEl.textContent = displayLabel;
      statusBannerEl.className = "status-banner " + (snap.state === "IDLE" ? "NO_FACE" : snap.state);
      videoCardEl.className = "video-card" + (snap.state !== "IDLE" ? " state-" + snap.state : "");

      leftEarEl.textContent = snap.leftEar !== null ? snap.leftEar.toFixed(3) : "—";
      rightEarEl.textContent = snap.rightEar !== null ? snap.rightEar.toFixed(3) : "—";
      closureDurationEl.textContent = snap.closureSec.toFixed(2) + " s";
      fpsEl.textContent = currentFps.toFixed(1) + " FPS";

      if (snap.subnote) {
        hudSubnoteEl.textContent = snap.subnote;
        hudSubnoteEl.classList.add("visible");
      } else {
        hudSubnoteEl.classList.remove("visible");
      }

      if (wasActiveReset) {
        alarm.resetCooldown();
      }
    }

    // -------------------------------------------------------------------------
    // Main requestAnimationFrame Processing Loop (Throttled to Target FPS)
    // -------------------------------------------------------------------------
    function startLoop(myToken) {
      lastVideoTime = -1;
      lastInferMs = 0;
      fpsCounter = 0;
      fpsWindowStartMs = performance.now();
      currentFps = 0;

      const tick = () => {
        if (!isRunning || myToken !== sessionToken) return;

        const nowMs = performance.now();
        const frameIntervalMs = 1000 / Math.max(1, config.targetFps);

        if (
          videoEl.readyState >= HTMLMediaElement.HAVE_CURRENT_DATA &&
          videoEl.videoWidth > 0 &&
          videoEl.currentTime !== lastVideoTime &&
          (nowMs - lastInferMs) >= frameIntervalMs
        ) {
          lastVideoTime = videoEl.currentTime;
          lastInferMs = nowMs;

          if (canvasEl.width !== videoEl.videoWidth || canvasEl.height !== videoEl.videoHeight) {
            canvasEl.width = videoEl.videoWidth;
            canvasEl.height = videoEl.videoHeight;
          }

          let results;
          try {
            results = faceLandmarker.detectForVideo(videoEl, nowMs);
          } catch (err) {
            console.error("Inference error:", err);
            if (activeDelegate === "GPU") {
              fallbackToCpuDelegate().catch((e) => showError("CPU fallback failed: " + e.message));
            }
            rafId = requestAnimationFrame(tick);
            return;
          }

          fpsCounter += 1;
          if (nowMs - fpsWindowStartMs >= 1000) {
            currentFps = (fpsCounter * 1000) / (nowMs - fpsWindowStartMs);
            fpsCounter = 0;
            fpsWindowStartMs = nowMs;
          }

          const landmarks = results?.faceLandmarks?.[0] || null;
          let leftMetrics = { valid: false, ear: null };
          let rightMetrics = { valid: false, ear: null };
          let yawAsymmetry = 0;

          if (landmarks) {
            leftMetrics = computeEyeMetrics(landmarks, LEFT_EYE, canvasEl.width, canvasEl.height);
            rightMetrics = computeEyeMetrics(landmarks, RIGHT_EYE, canvasEl.width, canvasEl.height);
            yawAsymmetry = computeYawAsymmetry(landmarks, canvasEl.width, canvasEl.height);

            if (calibrationActive) {
              if (leftMetrics.valid && rightMetrics.valid && yawAsymmetry <= config.maxYawAsymmetry) {
                calibrationSamples.push((leftMetrics.ear + rightMetrics.ear) * 0.5);
              }
              if (nowMs >= calibrationEndMs) {
                finishCalibration();
              }
            }
          }

          const prevState = classifier.state;
          const snap = classifier.update({
            nowMs,
            faceFound: Boolean(landmarks),
            leftMetrics,
            rightMetrics,
            yawAsymmetry
          });

          drawOverlay(landmarks, snap.state);

          const recoveredToActive =
            snap.state === "ACTIVE" && (prevState === "SLEEPING" || prevState === "DROWSY");
          updateUiFromSnapshot(snap, recoveredToActive);

          if (snap.state === "SLEEPING") {
            alarm.triggerIfDue(nowMs, false);
          } else if (snap.state === "NO_FACE") {
            alarm.stopImmediate();
          }
        }

        rafId = requestAnimationFrame(tick);
      };

      rafId = requestAnimationFrame(tick);
    }

    // -------------------------------------------------------------------------
    // Camera Start / Stop Lifecycle
    // -------------------------------------------------------------------------
    async function startCamera() {
      if (isRunning || isStarting) return;
      clearError();
      isStarting = true;
      const myToken = ++sessionToken;

      startBtn.disabled = true;
      startBtn.textContent = "Starting...";

      // Unlock Web Audio synchronously inside user click gesture.
      alarm.unlock();
      alarm.updateBadge();

      try {
        if (!navigator.mediaDevices || !navigator.mediaDevices.getUserMedia) {
          throw new Error(
            "Camera API (navigator.mediaDevices.getUserMedia) is unavailable. " +
            "Ensure you opened the HTTPS public link (not plain HTTP)."
          );
        }

        await ensureFaceLandmarker();
        if (myToken !== sessionToken) return;

        const stream = await navigator.mediaDevices.getUserMedia({
          video: {
            facingMode: "user",
            width: { ideal: 960 },
            height: { ideal: 540 },
            frameRate: { ideal: 30 }
          },
          audio: false
        });

        if (myToken !== sessionToken) {
          stream.getTracks().forEach((t) => t.stop());
          return;
        }

        mediaStream = stream;
        stream.getVideoTracks().forEach((track) => {
          track.addEventListener("ended", () => {
            if (myToken === sessionToken && isRunning) {
              stopCamera("Camera stream ended or disconnected.");
            }
          });
        });

        videoEl.srcObject = stream;
        await videoEl.play();
        if (myToken !== sessionToken) return;

        classifier.reset();
        alarm.resetCooldown();
        isRunning = true;
        isStarting = false;

        placeholderEl.classList.add("hidden");
        startBtn.textContent = "Start Camera";
        startBtn.disabled = true;
        stopBtn.disabled = false;
        calibrateBtn.disabled = false;

        startLoop(myToken);
      } catch (err) {
        stopCamera();
        let msg = err?.message || String(err);
        if (err?.name === "NotAllowedError" || err?.name === "PermissionDeniedError") {
          msg = "Camera permission was denied. Allow webcam access in your browser address bar and click Start Camera again.";
        } else if (err?.name === "NotFoundError" || err?.name === "DevicesNotFoundError") {
          msg = "No webcam device was found on this computer.";
        } else if (err?.name === "NotReadableError" || err?.name === "TrackStartError") {
          msg = "Your webcam is already in use by another application or browser tab.";
        }
        showError(msg);
      }
    }

    function stopCamera(reasonMessage = "") {
      sessionToken += 1;
      isRunning = false;
      isStarting = false;
      calibrationActive = false;

      if (rafId !== null) {
        cancelAnimationFrame(rafId);
        rafId = null;
      }

      if (mediaStream) {
        mediaStream.getTracks().forEach((track) => {
          try { track.stop(); } catch {}
        });
        mediaStream = null;
      }

      videoEl.srcObject = null;
      ctx.clearRect(0, 0, canvasEl.width, canvasEl.height);
      alarm.stopImmediate();
      alarm.resetCooldown();
      classifier.reset();
      currentFps = 0;

      placeholderEl.classList.remove("hidden");
      startBtn.disabled = false;
      startBtn.textContent = "Start Camera";
      stopBtn.disabled = true;
      calibrateBtn.disabled = true;
      calibrateBtn.textContent = "Calibrate Open Eyes";

      updateUiFromSnapshot({
        state: "IDLE",
        leftEar: null,
        rightEar: null,
        closureSec: 0,
        subnote: ""
      }, false);

      if (reasonMessage) {
        showError(reasonMessage);
      }
    }

    // -------------------------------------------------------------------------
    // Calibration & Settings Wiring
    // -------------------------------------------------------------------------
    function syncSlidersFromConfig() {
      earClosedSlider.value = config.earClosedThreshold;
      earDrowsySlider.value = config.earDrowsyThreshold;
      drowsyDurSlider.value = config.drowsyDurationSec;
      sleepDurSlider.value = config.sleepDurationSec;
      targetFpsSlider.value = config.targetFps;
      cooldownSlider.value = config.alarmCooldownSec;

      earClosedValEl.textContent = config.earClosedThreshold.toFixed(2);
      earDrowsyValEl.textContent = config.earDrowsyThreshold.toFixed(2);
      drowsyDurValEl.textContent = config.drowsyDurationSec.toFixed(1) + " s";
      sleepDurValEl.textContent = config.sleepDurationSec.toFixed(1) + " s";
      targetFpsValEl.textContent = config.targetFps + " FPS";
      cooldownValEl.textContent = config.alarmCooldownSec.toFixed(1) + " s";
    }

    function startCalibration() {
      if (!isRunning || calibrationActive) return;
      calibrationActive = true;
      calibrationSamples = [];
      calibrationEndMs = performance.now() + 1800;
      calibrateBtn.disabled = true;
      calibrateBtn.textContent = "Keep Eyes Open...";
    }

    function finishCalibration() {
      calibrationActive = false;
      calibrateBtn.disabled = !isRunning;
      calibrateBtn.textContent = "Calibrate Open Eyes";

      if (calibrationSamples.length < 5) {
        showError("Calibration did not capture enough valid eye samples. Face the camera and try again.");
        return;
      }

      const avgOpen = calibrationSamples.reduce((a, b) => a + b, 0) / calibrationSamples.length;
      config.earDrowsyThreshold = Number(clamp(avgOpen * 0.78, 0.18, 0.30).toFixed(2));
      config.earClosedThreshold = Number(clamp(avgOpen * 0.62, 0.12, config.earDrowsyThreshold - 0.03).toFixed(2));
      syncSlidersFromConfig();
      clearError();
    }

    earClosedSlider.addEventListener("input", () => {
      config.earClosedThreshold = parseFloat(earClosedSlider.value);
      if (config.earDrowsyThreshold < config.earClosedThreshold + 0.02) {
        config.earDrowsyThreshold = Number((config.earClosedThreshold + 0.02).toFixed(2));
      }
      syncSlidersFromConfig();
    });

    earDrowsySlider.addEventListener("input", () => {
      config.earDrowsyThreshold = parseFloat(earDrowsySlider.value);
      if (config.earClosedThreshold > config.earDrowsyThreshold - 0.02) {
        config.earClosedThreshold = Number((config.earDrowsyThreshold - 0.02).toFixed(2));
      }
      syncSlidersFromConfig();
    });

    drowsyDurSlider.addEventListener("input", () => {
      config.drowsyDurationSec = parseFloat(drowsyDurSlider.value);
      if (config.sleepDurationSec < config.drowsyDurationSec + 0.4) {
        config.sleepDurationSec = Number((config.drowsyDurationSec + 0.4).toFixed(1));
      }
      syncSlidersFromConfig();
    });

    sleepDurSlider.addEventListener("input", () => {
      config.sleepDurationSec = parseFloat(sleepDurSlider.value);
      if (config.sleepDurationSec < config.drowsyDurationSec + 0.4) {
        config.drowsyDurationSec = Number(Math.max(0.5, config.sleepDurationSec - 0.4).toFixed(1));
      }
      syncSlidersFromConfig();
    });

    targetFpsSlider.addEventListener("input", () => {
      config.targetFps = parseInt(targetFpsSlider.value, 10);
      syncSlidersFromConfig();
    });

    cooldownSlider.addEventListener("input", () => {
      config.alarmCooldownSec = parseFloat(cooldownSlider.value);
      syncSlidersFromConfig();
    });

    mirrorVideoCheck.addEventListener("change", () => {
      videoEl.classList.toggle("mirrored", mirrorVideoCheck.checked);
      canvasEl.classList.toggle("mirrored", mirrorVideoCheck.checked);
    });

    showOverlayCheck.addEventListener("change", () => {
      if (!showOverlayCheck.checked) {
        ctx.clearRect(0, 0, canvasEl.width, canvasEl.height);
      }
    });

    resetDefaultsBtn.addEventListener("click", () => {
      Object.assign(config, DEFAULTS);
      syncSlidersFromConfig();
    });

    startBtn.addEventListener("click", startCamera);
    stopBtn.addEventListener("click", () => stopCamera());
    muteBtn.addEventListener("click", () => {
      alarm.unlock();
      alarm.setMuted(!alarm.muted);
    });
    testAlarmBtn.addEventListener("click", () => {
      alarm.unlock();
      alarm.triggerIfDue(performance.now(), true);
    });
    calibrateBtn.addEventListener("click", startCalibration);

    window.addEventListener("beforeunload", () => stopCamera());

    // Initial UI state
    syncSlidersFromConfig();
    alarm.updateBadge();
  </script>
</body>
</html>
"""

INDEX_FILE.write_text(APP_HTML, encoding="utf-8", newline="\n")
print(f"Web application written to: {INDEX_FILE} ({INDEX_FILE.stat().st_size:,} bytes)")


class DrowsinessRequestHandler(http.server.SimpleHTTPRequestHandler):
    """Serves the self-contained MediaPipe web app and a lightweight /health check."""

    def do_GET(self):
        if self.path == "/health":
            body = b'{"status":"ok"}'
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(body)))
            self.send_header("Cache-Control", "no-store")
            self.end_headers()
            self.wfile.write(body)
            return
        if self.path == "/" or self.path.startswith("/?"):
            self.path = "/index.html"
        return super().do_GET()

    def end_headers(self):
        self.send_header("Cache-Control", "no-store")
        super().end_headers()

    def log_message(self, format, *args):
        # Keep Colab cell output clean while the browser runs inference locally.
        pass


class ReusableThreadingHTTPServer(socketserver.ThreadingMixIn, http.server.HTTPServer):
    allow_reuse_address = True
    daemon_threads = True


def find_available_port(preferred_port=8000):
    for port in [preferred_port] + list(range(8001, 8020)):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            if sock.connect_ex(("127.0.0.1", port)) != 0:
                return port
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return sock.getsockname()[1]


def start_local_server(directory: Path, preferred_port=8000):
    port = find_available_port(preferred_port)
    handler = functools.partial(DrowsinessRequestHandler, directory=str(directory))
    server = ReusableThreadingHTTPServer(("127.0.0.1", port), handler)
    thread = threading.Thread(target=server.serve_forever, daemon=True)
    thread.start()
    return server, port

In [ ]:
# Step 3: Launch the web server and expose it via a public HTTPS Cloudflare Quick Tunnel

import os
import re
import shutil
import subprocess
import sys
import time
import urllib.request
from collections import deque

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = None
    display = None

# Clean up any server/tunnel left running from an earlier execution of this cell.
if globals().get("_active_tunnel_proc") is not None:
    try:
        _active_tunnel_proc.terminate()
        _active_tunnel_proc.wait(timeout=3)
    except Exception:
        pass
    _active_tunnel_proc = None

if globals().get("_active_http_server") is not None:
    try:
        _active_http_server.shutdown()
        _active_http_server.server_close()
    except Exception:
        pass
    _active_http_server = None

if "start_local_server" not in globals() or not INDEX_FILE.exists():
    raise RuntimeError("Web app not initialized. Please run Step 2 first.")

CLOUDFLARED_PATH = shutil.which("cloudflared") or "/usr/local/bin/cloudflared"
if not os.path.exists(CLOUDFLARED_PATH):
    raise FileNotFoundError("cloudflared binary not found. Please run Step 1 (Setup) first.")

server, PORT = start_local_server(APP_DIR, preferred_port=8000)
_active_http_server = server

# Verify local server health before creating the public tunnel.
local_health_url = f"http://127.0.0.1:{PORT}/health"
with urllib.request.urlopen(local_health_url, timeout=5) as resp:
    if resp.status != 200:
        raise RuntimeError(f"Local web server health check failed with status {resp.status}")

print(f"Local web server running at http://127.0.0.1:{PORT}")
print("Starting Cloudflare Quick Tunnel for public HTTPS access...")

tunnel_proc = subprocess.Popen(
    [
        CLOUDFLARED_PATH,
        "tunnel",
        "--url",
        f"http://127.0.0.1:{PORT}",
        "--no-autoupdate",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
_active_tunnel_proc = tunnel_proc

public_url = None
recent_logs = deque(maxlen=40)
url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
start_wait = time.time()
TUNNEL_TIMEOUT_SEC = 30

while time.time() - start_wait < TUNNEL_TIMEOUT_SEC:
    if tunnel_proc.poll() is not None:
        break
    line = tunnel_proc.stdout.readline()
    if line:
        clean_line = line.strip()
        recent_logs.append(clean_line)
        match = url_pattern.search(clean_line)
        if match:
            public_url = match.group(0)
            break
    else:
        time.sleep(0.1)

if not public_url:
    tunnel_logs = "\n".join(recent_logs) or "(no output captured)"
    # Provide Colab port-proxy fallback if running inside Google Colab.
    fallback_url = None
    if "google.colab" in sys.modules:
        try:
            from google.colab import output as colab_output
            fallback_url = colab_output.eval_js(f"google.colab.kernel.proxyPort({PORT})")
        except Exception:
            fallback_url = None

    if fallback_url:
        print("Cloudflare Quick Tunnel did not return a URL within 30s.")
        print("Cloudflare logs:\n" + tunnel_logs)
        print("\nUsing Google Colab HTTPS proxy fallback instead:")
        print(fallback_url)
        public_url = fallback_url
    else:
        tunnel_proc.terminate()
        server.shutdown()
        server.server_close()
        raise RuntimeError(
            "Failed to obtain a public HTTPS URL from Cloudflare Quick Tunnel.\n"
            f"Recent cloudflared output:\n{tunnel_logs}"
        )

# Brief DNS propagation check for trycloudflare.com so the user's first click works immediately.
if "trycloudflare.com" in public_url:
    for _ in range(10):
        try:
            with urllib.request.urlopen(f"{public_url}/health", timeout=3) as resp:
                if resp.status == 200:
                    break
        except Exception:
            time.sleep(1.0)

print("\n" + "=" * 68)
print("READY! Open the public HTTPS link below in your browser:")
print(public_url)
print("=" * 68)
print("1. Click the HTTPS link above (opens in a new tab).")
print("2. Click 'Start Camera' and allow browser webcam permission.")
print("3. Keep this Colab cell running while using the detector.")
print("4. Stop this cell (Stop button) when you are finished.\n")

if display is not None and HTML is not None:
    display(
        HTML(
            f'<div style="padding:12px 16px;border-radius:10px;background:#0f172a;border:1px solid #3b82f6;color:#f8fafc;font-family:sans-serif;">'
            f'<strong>Public HTTPS App URL:</strong> '
            f'<a href="{public_url}" target="_blank" rel="noopener noreferrer" '
            f'style="color:#60a5fa;font-size:1.05em;font-weight:bold;">{public_url}</a>'
            f'<div style="margin-top:6px;font-size:0.9em;color:#cbd5e1;">Open the link above, click <b>Start Camera</b>, and allow webcam permission.</div>'
            f"</div>"
        )
    )

try:
    # Drain tunnel output in a daemon thread so pipe buffers never fill and Ctrl+C responds immediately.
    import threading as _threading

    def _drain_tunnel_logs():
        try:
            for _line in tunnel_proc.stdout:
                recent_logs.append(_line.strip())
        except Exception:
            pass

    _threading.Thread(target=_drain_tunnel_logs, daemon=True).start()

    while True:
        if tunnel_proc.poll() is not None and "trycloudflare.com" in public_url:
            raise RuntimeError(
                "cloudflared tunnel process exited unexpectedly.\n"
                + "\n".join(recent_logs)
            )
        time.sleep(0.5)
except KeyboardInterrupt:
    print("\nShutting down server and tunnel...")
finally:
    if tunnel_proc.poll() is None:
        tunnel_proc.terminate()
        try:
            tunnel_proc.wait(timeout=5)
        except Exception:
            tunnel_proc.kill()
    server.shutdown()
    server.server_close()
    _active_tunnel_proc = None
    _active_http_server = None
    print("Cleanly stopped.")

## Usage, Calibration & Troubleshooting

### How Eye Aspect Ratio (EAR) & Classification Work
- **EAR Formula:** Using MediaPipe's 478-point face mesh (`@mediapipe/tasks-vision@1.2.3`), each eye is measured from its pixel-scaled horizontal corner span and three vertical eyelid pairs (Left eye: outer `263`, inner `362`, vertical `[387,373]`, `[386,374]`, `[385,380]`; Right eye: outer `33`, inner `133`, vertical `[160,144]`, `[159,145]`, `[158,153]`).
- **Both Eyes Required:** Both eyes must agree before accumulating closure time, preventing a single-eye occlusion or landmark glitch from triggering `SLEEPING`.
- **Time-Based Thresholds (Independent of FPS):**
  - **ACTIVE (Green):** Face detected, eyes open above Drowsy EAR threshold.
  - **DROWSY (Orange):** Both eyes stay below `Drowsy EAR` (`0.24` default) for `>= 1.2 s` (safely above normal `0.15–0.40 s` blinks).
  - **SLEEPING (Red + Alarm):** Both eyes stay below `Closed EAR` (`0.19` default) for `>= 2.5 s`.
  - **NO FACE (Grey):** No face detected for `>= 0.5 s` (brief occlusions are tolerated without flickering).
- **Calibrating for Your Eyes / Glasses:** Start the camera, look straight ahead with eyes naturally open, and click **Calibrate Open Eyes** (or adjust the sliders directly).

### Verification Checklist
1. Open the public `https://...` URL and click **Start Camera** -> allow webcam permission.
2. Keep both eyes open -> HUD banner shows **ACTIVE** (Green) immediately (no stuck `Monitoring...`).
3. Blink naturally -> status stays **ACTIVE**.
4. Half-close eyes for `> 1.2 s` -> status changes to **DROWSY** (Orange).
5. Close both eyes for `> 2.5 s` -> status changes to **SLEEPING** (Red) and plays the alarm once per cooldown window; opening eyes resets the alarm latch.
6. Move out of frame -> status changes to **NO FACE** (Grey) after `0.5 s`.
7. Click **Stop Camera**, then **Start Camera** again -> webcam indicator turns off on Stop and restarts cleanly with no duplicate FPS/loops.

### Troubleshooting
- **Camera blocked or unavailable:** Verify you opened the **`https://`** link (browsers block `getUserMedia` on plain `http://` except `localhost`) and grant camera permission in the browser address bar.
- **Model failed to load:** The browser fetches `@mediapipe/tasks-vision` from `cdn.jsdelivr.net` and `face_landmarker.task` from `storage.googleapis.com`. Disable strict script blockers for this tab if needed.
- **Slow FPS / no WebGL:** The app automatically falls back to CPU delegate if WebGL GPU is unavailable. Lower **Inference Rate Cap** (e.g., `10 FPS`) on older laptops.
- **Tunnel stopped:** Re-run **Step 3**; any previous server/tunnel is cleaned up automatically.